## 목표

---

이번 실습에서는 LangChain으로 개발한 RAG를 다음 블로그의 정보와 연동합니다.

요구사항은 다음과 같습니다.

- [ ]  RAG internet source를 https://spartacodingclub.kr/blog/all-in-challenge_winner 로 설정합니다.
    - RAG에서 활용할 source로 위의 링크를 전달합니다.
    - 사이트가 달라졌기 때문에 이전 실습 코드와 다르게 load 해야 합니다. 어디를 어떻게 수정해야 할지 고민해보도록 합시다.
    - LLM은 GPT를 사용하시면 됩니다. 모델은 `gpt-4o-mini`로 설정하시면 됩니다.
- [ ]  GPT에게 `“ALL-in 코딩 공모전 수상작들을 요약해줘.”`를 물은 뒤의 답변을 출력합니다.

In [1]:
!pip install langchain-community langchain-chroma langchain-openai bs4 chardet

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 3.2 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 35.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.4/62.4 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 611.1/611.1 kB 35.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 56.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 57.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 284.2/284.2 kB 23.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.2/95.2 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 64.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 4.1 MB/s eta 0:00:00
   

In [2]:
import bs4
from langchain import hub
from langchain_chroma import Chroma
from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [3]:
from dotenv import load_dotenv
import os

# openai_key = ""
load_dotenv()  # .env 파일 로드
openai_key = os.getenv("OPENAI_API_KEY")  # 키 가져오기


In [4]:
llm = ChatOpenAI(model="gpt-4o-mini", api_key=openai_key)

In [18]:
import requests
from bs4 import BeautifulSoup

url = "https://spartacodingclub.kr/blog/all-in-challenge_winner"
res = requests.get(url)

try:
    # 1차 시도: 감지된 인코딩
    import chardet
    detected = chardet.detect(res.content)
    encoding = detected['encoding']
    html = res.content.decode(encoding)
except UnicodeDecodeError:
    # 실패 시 UTF-8로 fallback
    print(f"⚠️ {encoding} 디코딩 실패 → UTF-8 재시도")
    html = res.content.decode("utf-8", errors="ignore")

# 파싱
soup = BeautifulSoup(html, "html.parser")
main = soup.select_one(".editedContent")
if not main:
    raise ValueError("❌ .editedContent 클래스를 찾을 수 없습니다.")
text = main.get_text(separator="\n")

# 미리보기
print(text[:1000])


⚠️ Windows-1254 디코딩 실패 → UTF-8 재시도
코딩은 더 이상 개발자만의 영역이 아닙니다. 누구나 아이디어만 있다면 창의적인 서비스를 만들어 세상을 바꿀 수 있습니다. 스파르타코딩클럽에서는 이러한 가능성을 믿고, 누구나 코딩을 통해 자신의 아이디어를 실현하고 실제 문제를 해결하는 경험을 쌓을 수 있도록 다양한 프로그램을 마련하고 있습니다.
<All-in> 코딩 공모전은 대학생들이 캠퍼스에서 겪은 불편함과 문제를 자신만의 아이디어로 해결해보는 대회였는데요. 이번 공모전에서 다양한 혁신적인 아이디어와 열정으로 가득한 수많은 프로젝트가 탄생했습니다. 그중 뛰어난 성과를 낸 수상작 6개를 소개합니다.
🏆 대상
[Lexi Note] 언어공부 필기 웹 서비스
서비스 제작자: 다나와(김다애, 박나경)
💡W는 어문학을 전공하는 대학생입니다. 매일 새로운 단어와 문장 구조를 공부하고 있지만, 효율적으로 학습하는 것이 쉽지 않았습니다. 단어의 의미를 찾기 위해 사전을 뒤적이고, 긴 문장을 이해하려고 번역기를 사용하다 보면, 필기 노트는 어느새 뒷전으로 밀려났거든요. 사전, 번역기, 원서, 필기노트를 왔다 갔다 하다 보면 시간이 다 지나가 버리곤 했죠.
W와 같이 어문 전공생은 문법, 어휘, 문장 구조 등 다양한 자료를 학습해야 합니다. 여러 자료를 번갈아 학습하다보니 ‘사전-번역기-원서-필기노트’ 왕복으로 학습 효율이 나지 않아 고민인 경우도 많으실 거예요. <Lexi Note>는 단어를 드래그하면 네이버 사전으로 바로 연동 돼 단어의 의미를 찾으며 동시에 필기 할 수 있어요. 이외에도 번역 버튼을 누르면 파파고 번역기가 연동돼 긴 문장도 쉽게 이해할 수 있어요. 언어 학습에 필요한 할일 목록과 스케줄 템플릿을 제공하여 효율적으로 공부할 수 있습니다. 필기, 사전, 번역을 한번에 쉽고 편하게 이용할 수 있죠. 더 이상 시간 낭비 없이 효율적으로 어문학을 공부하며 학습 속도도 눈에 띄게 빨라질 수 있어요. 언어 공부의 복잡함을 단순하게 만들어주는 Lexi Note

In [24]:
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

doc = Document(page_content=text)
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
splits = splitter.split_documents([doc])

In [32]:
vectorstore = Chroma.from_documents(
    documents=splits,
    embedding=OpenAIEmbeddings(api_key=temp_key)
)
retriever = vectorstore.as_retriever(search_kwargs={"k": 6})  # 최대 6개까지 받기



query = f"""다음은 ALL-in 코딩 공모전 수상작에 대한 소개입니다:

{text}

위 텍스트를 바탕으로 다음과 같이 정리해줘:

1. 수상명 (대상/우수상/입선)
2. 서비스명
3. 핵심 기능 또는 해결한 문제

수상작이 6개이므로 모두 개별 항목으로 정리해줘.
."""
retrieved_docs = retriever.invoke(query)

# 프롬프트 템플릿 로딩
prompt = hub.pull("rlm/rag-prompt")
user_prompt = prompt.invoke({
    "context": "\n\n".join(doc.page_content for doc in retrieved_docs),
    "question": query
})


/usr/local/lib/python3.11/dist-packages/langsmith/client.py:280: LangSmithMissingAPIKeyWarning: API key must be provided when using hosted LangSmith API
  warnings.warn(


In [33]:
# 4. GPT에게 질문하고 결과 출력
response = llm.invoke(user_prompt)
print("🧠 GPT 요약 결과:")
print(response.content)


🧠 GPT 요약 결과:
1. 대상  
   서비스명: [Lexi Note]  
   핵심 기능: 사전과 번역기와 연동되어 필기를 효율적으로 할 수 있도록 도와줌.  

2. 우수상  
   서비스명: [우리집 히어로즈]  
   핵심 기능: 벌레 퇴치 요청과 히어로 매칭 서비스로 안전한 환경에서 문제를 해결함.  

3. 우수상  
   서비스명: [에코 클래스룸]  
   핵심 기능: 익명으로 학생들의 의견이나 질문을 제출하여 교수와의 소통을 강화함.  

4. 입선  
   서비스명: [Crewing]  
   핵심 기능: 대학생들이 적합한 연합 동아리를 쉽게 찾고 가입할 수 있도록 지원함.  

5. 입선  
   서비스명: [학교생활 매니저]  
   핵심 기능: 일정, 과제, 성적 등을 통합 관리하여 학교 생활을 효율적으로 도와줌.  

6. 입선  
   서비스명: [BLOTIE]  
   핵심 기능: 외국인과 한국인 학생 간의 교류를 위한 매칭 플랫폼 제공.  
